In [1]:
# Bates call price = Heston stochastic volatility + jumps. Needs only numpy. Everything annual and decimal.
#   S, K spot and strike (K can be an array) | T years to expiry | r, q risk-free rate and dividend yield (FX: r = domestic, q = foreign)
#   v0, theta  current and long-run VARIANCE (volatility squared, e.g. 0.16 = 40% vol)   kappa  speed of reversion of the variance
#   xi  vol of variance   rho  stock/variance correlation   lam  jumps per year   jump_mean  average % jump E[Y-1]   jump_std  std of the log jump
# lam=0 gives Heston. Put price: call - S*exp(-q*T) + K*exp(-r*T).
import numpy as np
from numpy.polynomial.legendre import leggauss

def bates_call(S, K, T, r, v0, theta, kappa, xi, rho, lam, jump_mean, jump_std, q=0.0):
    K = np.atleast_1d(np.asarray(K, float)); m, sj = jump_mean, jump_std; xi = max(xi, 1e-6); mu = np.log1p(m) - 0.5 * sj ** 2
    def cf(u):                                    # characteristic function of ln(S_T) under the risk-neutral measure
        b = kappa - 1j * rho * xi * u; d = np.sqrt(b * b + xi ** 2 * (1j * u + u * u)); g = (b - d) / (b + d); e = np.exp(-d * T)
        C = kappa * theta / xi ** 2 * ((b - d) * T - 2 * np.log((1 - g * e) / (1 - g))); D = (b - d) / xi ** 2 * (1 - e) / (1 - g * e)
        jumps = lam * T * (np.exp(1j * u * mu - 0.5 * u * u * sj ** 2) - 1 - 1j * u * m)      # compensated jumps
        return np.exp(1j * u * (np.log(S) + (r - q) * T) + C + D * v0 + jumps)
    x, w = leggauss(400); u = 100 * (x + 1) + 1e-10; w = 100 * w; lnK = np.log(K)[:, None]; U = u[None, :]   # integrate over u in (0, 200)
    P2 = 0.5 + (w * np.real(np.exp(-1j * U * lnK) * cf(U) / (1j * U))).sum(1) / np.pi
    P1 = 0.5 + (w * np.real(np.exp(-1j * U * lnK) * cf(U - 1j) / (S * np.exp((r - q) * T)) / (1j * U))).sum(1) / np.pi
    price = S * np.exp(-q * T) * P1 - K * np.exp(-r * T) * P2
    return price if price.size > 1 else float(price[0])
